# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. Выведет: 1, 1.33 (и т.д.), isna() выводит series из булевых значений где true это nan, а fillna() заполняет вместо nan то значение которые мы напишем в скобки. 
2. Выведет 2. Сначала убираются пробелы по бокам, потом все значение приводится к нижнему регистру, а потом посчитает количество уникальных значений.
3. Выведет 3, так как мы не сохранили изменения
4. Выведет 1005. Сначала список превращается в Series, потом ко всем элементам выполняется функция replace (убираем все пробелы), после этого все строки превращаются в float и мы сумируем все значения.
5. Выведет [9, 10]. Значения из Series превращаем в объект Timestamp и перетаскиваем в переменную d, из d мы достаём месяцы и превращаем в список.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [31]:
# 1
import numpy as np
import pandas as pd
def clean(path):
    path = pd.read_csv("orders.csv")
    CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
    
    path["amount"] = pd.to_numeric(path["amount"].str.replace(" ", ""))
    print(f"Осталось строк: {len(path)}")
    path["city"] = path["city"].str.replace(" ", "").str.lower().map(CITY)
    print(f"Осталось строк: {len(path)}")
    path = path.drop_duplicates()
    print(f"Осталось строк: {len(path)}")
    path["order_date"] = pd.to_datetime(path["order_date"])
    print(f"Осталось строк: {len(path)}")
    path = path[path["customer_id"] != "TEST"]
    print(f"Осталось строк: {len(path)}")
    print(f"Сколько у amount NaN: {path["amount"].isna().sum()}, Сколько у city NaN: {path["city"].isna().sum()}, Уникален ли order_id: {path["order_id"].is_unique}")
    return path

a = pd.DataFrame()
beta_test = clean(a)
beta_test

Осталось строк: 5463
Осталось строк: 5463
Осталось строк: 5283
Осталось строк: 5283
Осталось строк: 5280
Сколько у amount NaN: 0, Сколько у city NaN: 0, Уникален ли order_id: True


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен
5461,103460,2026-10-31 22:00:00,C02162,Алматы,Электроника,24670.0,2.0,отменён


In [2]:
import numpy as np
import pandas as pd
d = pd.read_csv("orders.csv")
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

d["amount"] = pd.to_numeric(d["amount"].str.replace(" ", ""))
d["city"] = d["city"].str.replace(" ", "").str.lower().map(CITY)
d["city"].value_counts()

city
Алматы       2188
Астана       1390
Шымкент       804
Караганда     538
Актобе        397
Атырау        146
Name: count, dtype: int64

In [11]:
d = d.drop_duplicates()


np.int64(0)

In [21]:
d["order_date"] = pd.to_datetime(d["order_date"])
d = d[d["customer_id"] != "TEST"]
(d["customer_id"] == "TEST").sum()

np.int64(0)

In [27]:
d["order_id"].is_unique

True

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [62]:
# 2
beta_test["month"] = beta_test["order_date"].dt.month
sep = beta_test["month"] == 9
octo = ~sep
paid = beta_test["status"] == "оплачен"
revenue_sep = beta_test.loc[sep & paid, "amount"].sum()
revenue_octo = beta_test.loc[octo & paid, "amount"].sum()
print(f"Прибыль за сентябрь: {revenue_sep} тг, Прибыль за октябрь: {revenue_octo} тг, Изменение: {revenue_octo / revenue_sep - 1:.2%} %")

for i in beta_test["city"].unique():
    print(f"{i} прибыль за сентябрь: {beta_test.loc[sep & paid & (beta_test["city"] == i), "amount"].sum()} тг , а за октябрь: {beta_test.loc[octo & paid & (beta_test["city"] == i), "amount"].sum()} тг, изменение: {beta_test.loc[octo & paid & (beta_test["city"] == i), "amount"].sum() / beta_test.loc[sep & paid & (beta_test["city"] == i), "amount"].sum() - 1:.2%} %")

Прибыль за сентябрь: 35243410.0 тг, Прибыль за октябрь: 34419520.0 тг, Изменение: -2.34% %
Алматы прибыль за сентябрь: 14501070.0 тг , а за октябрь: 14643020.0 тг, изменение: 0.98% %
Астана прибыль за сентябрь: 9034220.0 тг , а за октябрь: 8850530.0 тг, изменение: -2.03% %
Караганда прибыль за сентябрь: 3648520.0 тг , а за октябрь: 3371680.0 тг, изменение: -7.59% %
Шымкент прибыль за сентябрь: 4558650.0 тг , а за октябрь: 4044010.0 тг, изменение: -11.29% %
Атырау прибыль за сентябрь: 923990.0 тг , а за октябрь: 969860.0 тг, изменение: 4.96% %
Актобе прибыль за сентябрь: 2576960.0 тг , а за октябрь: 2540420.0 тг, изменение: -1.42% %


После сравнения с домашкой номер 4, могу сказать что результаты по городу и их прибылью за месяц полностью совпадают. 
Среди всех городов, доля выручки сильнее всех падает у Шымкент. 

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

# 3
Я не до конца понял суть вопроса, но отвечу как я понял.
Если говорить относительно урока, то менеджер забыл почистить данные (убрать тестовые данные и дубликаты). Сильнее всего исказило наличие тестовых данных (TEST) на измерение доли выручки (с -22,7% до -8,2%)

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [83]:
# 4
print(f"Количество пропусков: {beta_test["delivery_days"].isna().sum()}")
print(beta_test.loc[beta_test["delivery_days"].isna(), "status"].value_counts())
print(beta_test.loc[beta_test["delivery_days"].isna(), "city"].value_counts())
SH = beta_test.loc[beta_test["city"] == "Шымкент", "delivery_days"]
print(f"dropna: {(SH.dropna() <= 2).mean():.2%}, fillna(медиана): {(SH.fillna(SH.median()) <= 2).mean():.2%}, fillna(0): {(SH.fillna(0) <= 2).mean():.2%}")

Количество пропусков: 158
status
оплачен    144
отменён     14
Name: count, dtype: int64
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
dropna: 29.84%, fillna(медиана): 28.99%, fillna(0): 31.83%


По моему мнение, самый частный вариант является fillna с медианой.
Если брать dropna, то он просто стирает данные из таблицы, уменьшая количество заказов, что при анализе даёт не совсем реальные данные. 
Если брать fillna(0), то тут уже более лучее можно рассчитать долю так как количество заказов не уменьшается, но брать все заказы как за доставку за 0 дней (моментально) будто бы тоже не совсем правильно. 
Если мы возьмём fillna с медианой, то результат более менее должен быть приближен к реальному результату, поскольку медиана не может обмануть при выбросах, так как берет среднее значение.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [84]:
# 5
df = pd.read_csv("orders.csv")
df = df.drop_duplicates()
df["city"] = df["city"].str.lower().str.strip().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


Первое что бросается глаза, то что AI не присвоил удаление дубликатов к DataFrame, что привело к тому что в нём остались дубликаты.
Вторая критическая ошибка заключается в том, что в колоне city он не удалил боковые пробелы, и то что он не использовал map, и у него нету словаря для приведения к единому виду городов.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [102]:
# 6
beta_test["day_name"] = beta_test["order_date"].dt.day_name()
print(beta_test["day_name"].value_counts())
print(f"Меньше всего заказов в {beta_test["day_name"].value_counts().idxmin()}. По сравнению с самым загруженным днём, у него {beta_test["day_name"].value_counts().min() / beta_test["day_name"].value_counts().max() - 1:.2%} загруженности")

day_name
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего заказов в Monday. По сравнению с самым загруженным днём, у него -12.98% загруженности


Это означает, что в данный день у них больше всего свободного времени по сравнению с другими днями недели, поэтому на этот день можно выделить меньше смен или плановые выходные.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [180]:
# 7
print(f"Средний срок доставки в сентябре составляет: {beta_test.loc[(beta_test["city"] == "Шымкент") & sep, "delivery_days"].mean().round(2)} дня")
print(f"Средний срок доставки в октябре составляет: {beta_test.loc[(beta_test["city"] == "Шымкент") & octo, "delivery_days"].mean().round(2)} дня")
last_good_day = beta_test.loc[(beta_test["city"] == "Шымкент") & (beta_test["delivery_days"] < 5)].sort_values("order_date").iloc[-1]["order_date"]
suspicious_date = beta_test.loc[(beta_test["city"] == "Шымкент") & (beta_test["delivery_days"] >= 5) & (beta_test["order_date"] > last_good_day)].sort_values("order_date").iloc[0]["order_date"]
print(f"Первая дата доставки, которая превысила доставку не меньше 5 дней: {suspicious_date}")

for i in beta_test["city"].unique():
    print(f"Средний срок доставки в {i} составляет {beta_test.loc[(beta_test["city"] == i) & (beta_test["order_date"] >= suspicious_date), "delivery_days"].mean()}")

Средний срок доставки в сентябре составляет: 2.03 дня
Средний срок доставки в октябре составляет: 6.54 дня
Первая дата доставки, которая превысила доставку не меньше 5 дней: 2026-10-01 10:00:00
Средний срок доставки в Алматы составляет 2.013435700575816
Средний срок доставки в Астана составляет 1.9953846153846153
Средний срок доставки в Караганда составляет 1.9914529914529915
Средний срок доставки в Шымкент составляет 6.5358024691358025
Средний срок доставки в Атырау составляет 2.013888888888889
Средний срок доставки в Актобе составляет 2.0974358974358975


После анализа стало понятно, что после данного времени: 2026-10-01 10:00:00, средний срок доставки увечилось до 6.5 дней. Необходимо проверить процессы в самом Шымкенте, возможно появились какие либо проблемы с выгрузкой или сборке заказов на складе.  

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [244]:
# 8
def to_amount(s):
    s = s.str.replace(" ", "").str.replace(",", ".")

    new_s = []
    for i in s.to_list():
        if i.replace(".", "").isdigit():
            new_s.append(float(i))
        else:
            new_s.append(np.nan)
    return pd.Series(new_s)

test = to_amount(pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"]))
test

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64

In [251]:
new_test = pd.read_csv("orders.csv")
check = to_amount(new_test["amount"])
check.sum()

np.float64(109883360.0)

In [252]:
new_test_1 = pd.read_csv("orders.csv")
pd.to_numeric(new_test_1["amount"].str.replace(" ", "")).sum()



np.float64(109883360.0)

In [198]:
a = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
a = a.str.replace(" ", "").str.replace(",", ".")
clean = a.str.replace(".", "").str.isdigit()
a.loc[clean]

0     True
1     True
2     True
3    False
dtype: bool

In [209]:
a.str.float()

AttributeError: 'StringMethods' object has no attribute 'float'

In [206]:
b = clean.replace(False, np.nan)
b

0    True
1    True
2    True
3     NaN
dtype: object

In [219]:
a.loc[clean][0]

'15990'

In [238]:
new_a = []
for i in a.to_list():
    if i.replace(".", "").isdigit():
        new_a.append(float(i))
    else:
        new_a.append(np.nan)


In [242]:
pd.Series(new_a)

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64

In [237]:
a.to_list()

['15990', '15990.0', '1200.50', 'нетданных']

In [243]:
a

0        15990
1      15990.0
2      1200.50
3    нетданных
dtype: str